---
title: Haplotype painting
---

## Model

In [2]:
import tspaint
from tspaint.sim import SOURCE_A, SOURCE_B, ADMIXED, ANCESTRAL
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import demesdraw
import msprime
from vscodenb import set_vscode_theme, vscode_theme
set_vscode_theme()

Ne=2_000
T_admix=200.0
T_split=20_000
f_A=0.3
sequence_length = 5e6
recombination_rate = 1e-8
mutation_rate = 1.1e-8
ploidy = 2
random_seed = 1
infer = True

n_admix = 8
n_ref = 8

d = msprime.Demography()
d.add_population(name=SOURCE_A, initial_size=Ne)
d.add_population(name=SOURCE_B, initial_size=Ne)
d.add_population(name=ADMIXED, initial_size=Ne)
d.add_population(name=ANCESTRAL, initial_size=Ne)
# Admixed pop forms (backward in time) as a mixture of the two sources.
d.add_admixture(time=T_admix, derived=ADMIXED,
                ancestral=[SOURCE_A, SOURCE_B], proportions=[f_A, 1.0 - f_A])
# Census after the pulse: every lineage is now in A or B; census nodes label
# each lineage's source per genomic segment (the local-ancestry truth).
d.add_census(time=T_admix + 1)
# Sources coalesce into a common ancestor deeper in time.
d.add_population_split(time=T_split, derived=[SOURCE_A, SOURCE_B],
                        ancestral=ANCESTRAL)
graph = msprime.Demography.to_demes(d)
with vscode_theme(style='ticks'):
    fig, ax = plt.subplots(figsize=(4, 3))
    demesdraw.tubes(graph, ax=ax, seed=1)

ImportError: numpy._core.multiarray failed to import

## Simulate

In [ ]:
ts = msprime.sim_ancestry(
        samples={ADMIXED: n_admix, SOURCE_A: n_ref, SOURCE_B: n_ref},
        demography=d,
        sequence_length=sequence_length,
        recombination_rate=recombination_rate,
        ploidy=ploidy,
        random_seed=random_seed,
    )

pop = ts.tables.nodes.population
name = {p: ts.population(p).metadata.get("name", str(p)) for p in range(ts.num_populations)}
A = next(p for p, n in name.items() if n == SOURCE_A)
B = next(p for p, n in name.items() if n == SOURCE_B)
admix = next(p for p, n in name.items() if n == ADMIXED)
sop = {A: 0, B: 1}
labels = {int(s): sop[pop[s]] for s in ts.samples() if pop[s] in (A, B)}
queries = [int(s) for s in ts.samples() if pop[s] == admix]
truth = tspaint.metrics.map_truth({q: tspaint.local_ancestry_truth(ts)[0][q] for q in queries}, sop)
work = ts
if infer:
    work = tspaint.io.tsinfer(
        tspaint.io.add_mutations(
            ts, rate=mutation_rate, random_seed=random_seed
            )
        )
    # work = tspaint.io.singer(
    #     tspaint.io.add_mutations(
    #         ts, rate=mutation_rate, random_seed=random_seed
    #         ),
    #         Ne=Ne, recombination_rate=recombination_rate, mutation_rate=mutation_rate
    #     )    


## Paint

In [ ]:
painting = tspaint.paint(ts, labels)

## Plot 

In [ ]:
#| label: fig-painting

sns.set_style("ticks")

L = ts.sequence_length
qs = painting.queries
segments = painting.segments(deadband=0.4)

sm = matplotlib.cm.ScalarMappable(norm=matplotlib.colors.Normalize(0, 1), cmap='coolwarm')
fig = plt.figure(figsize=(9, 0.3 * len(qs) + 1))
gs = fig.add_gridspec(len(qs), 2, width_ratios=[1,0.03], hspace=0)
axes = [fig.add_subplot(gs[i, 0]) for i in range(len(qs))]

for i, q in enumerate(qs):
    for (l, r, s) in truth[q]:
        axes[i].barh(-0.25, r - l, left=l, height=0.5,
                color=sm.to_rgba(1.0 if s == 0 else 0.0), edgecolor="none")
    for (l, r, s) in segments[q]:
        axes[i].barh(0.25, r - l, left=l, height=0.5,
                color=sm.to_rgba(1.0 if s == 0 else 0.0), edgecolor="none")
    for seg in painting.posteriors[q]:
        axes[i].barh(1, seg.right - seg.left, left=seg.left, height=1,
                color=sm.to_rgba(seg.posterior[0]), edgecolor="none")
    axes[i].set_ylim(-0.5, 1.5)
    axes[i].set_xlim(0, L)
    axes[i].set_ylabel('hapl. name', rotation=0, fontsize=7, color="1.0", horizontalalignment="right")
    axes[i].yaxis.set_major_locator(ticker.NullLocator())
    if i < len(axes)-1:
        axes[i].xaxis.set_major_locator(ticker.NullLocator())

ax = fig.add_subplot(gs[:, 1])
ax.set_axis_off()
cb = fig.colorbar(sm, ax=ax, fraction=0.5, pad=0.01)
cb.set_label("P(ancestry A)")

plt.tight_layout()

AttributeError: 'NoneType' object has no attribute 'family_name'

Error in callback <function _draw_all_if_interactive at 0x11ec680e0> (for post_execute), with arguments args (),kwargs {}:


AttributeError: 'NoneType' object has no attribute 'family_name'

AttributeError: 'NoneType' object has no attribute 'family_name'

AttributeError: 'NoneType' object has no attribute 'family_name'

<Figure size 900x580 with 18 Axes>

## Accuracy

Balanced accuracy and mean confidence of the soft painting.

In [ ]:
ba = tspaint.metrics.balanced_accuracy(painting.posteriors, truth, samples=queries)
conf = tspaint.metrics.mean_confidence(painting.posteriors, samples=queries)
print(f"balanced accuracy = {ba:.3f}   mean confidence = {conf:.3f}")

balanced accuracy = 0.947   mean confidence = 0.564
